## Parsing text

## Lecture objectives

1. Demonstrate how to parse unstructured text data

Let's start by loading in the data that we saved at the end of the previous lecture. We can read a pandas dataframe using the `read_pickle()` function. (If we had saved it as a .csv, we'd use `read_csv()`.)

In [1]:
import pandas as pd
smalldf = pd.read_pickle('../scratch/Seattle_permits.pandas')
smalldf.head()

,permitnum,permitclass,permitclassmapped,permittypemapped,description,statuscurrent,originaladdress1,originalcity,originalstate,originalzip,...,housingunitsremoved,housingunitsadded,applieddate,issueddate,expiresdate,decisiondate,permittypedesc,contractorcompanyname,estprojectcost,newdescription
0,3001212-LU,Single Family/Duplex,Residential,Master Use Permit,PROJECT CANCELLED 12/8/2010 -- This short plat...,Canceled,6519 S BANGOR ST,SEATTLE,WA,98178,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PROJECT CANCELLED 12/8/2010 -- This short plat...
1,3001271-LU,Single Family/Duplex,Residential,Master Use Permit,Land Use Permit to adjust the boundary between...,Completed,4226 1ST AVE NW,SEATTLE,WA,98107,...,0.0,0.0,2005-12-16,2006-05-15,2007-11-15,2006-05-10,NaN,NaN,NaN,Land Use Permit to adjust the boundary between...
2,3001310-LU,Single Family/Duplex,Residential,Master Use Permit,Land use application to adjust the boundary be...,Completed,941 23RD AVE S,SEATTLE,WA,98144,...,NaN,NaN,2007-02-14,2008-08-28,2011-08-14,2008-08-13,NaN,NaN,NaN,Land use application to adjust the boundary be...
3,3001312-LU,N/A,N/A,Master Use Permit,Cancelled due to no activity for more than 9 y...,Canceled,3131 E MADISON ST,SEATTLE,WA,98112,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cancelled due to no activity for more than 9 y...
4,3001440-LU,Commercial,Non-Residential,Master Use Permit,PROJECT CANCELLED 5/23/2011 -- Project On Hold...,Canceled,9030 13TH AVE NW,SEATTLE,WA,98117,...,NaN,NaN,2005-08-12,NaN,NaN,NaN,NaN,NaN,NaN,PROJECT CANCELLED 5/23/2011 -- Project On Hold...


We've already scraped the description for each project. 

Suppose we want to extract a particular piece of information? For example, how do we get the number of parking spaces? Well, that depends on whether the city uses consistent terminology. 

You'll need to design a set of rules that cover different possibilities. For example, the description might say "2 parking spaces" or "TWO PARKING SPACES" or "1 uncovered and 1 covered parking space." Looking at your data is key.

For starters, let's take the simplest case. We'll add a column to our dataframe that indicates whether there is "no parking" in the project description.

In [2]:
# import the numpy library, which underlies pandas
# we'll use its nan (null) value to indicate missing data
#numpyはNaNを入力（np.nan）できるようにするためにimportされている
#NaNの代わりに空文字列 '' や 'unknown' などを返す方法もあるが、それだと型がバラバラ（True/False/文字列）になって分析しづらく、欠損値として扱えない（.isna() では検出されない）
#なので、分析や統計処理を見越して np.nan を使うのがベストプラクティス 

import numpy as np

#説明文に「no parking」や「zero parking」が含まれていたらTrue、単に「parking」が含まれていたらFalse、それ以外はNaN（情報不足）にする
def noparking(description):
    # convert the description to lower case
    text = description.lower() #説明文をすべて小文字に変換。これにより`"No Parking"`や`"NO PARKING"`など大小文字の違いを気にせず処理できる
    if 'no parking' in text:
        return True
    elif 'zero parking' in text:
        return True
    elif 'parking' in text:
        return False
    else:
        # capture all other possibilities
        return np.nan

# Now apply our function
#smalldfの'description'列に対して、さっきのnoparking()関数を行ごとに適用（apply）し、結果を'noparking'という新しい列に保存
#"変数.apply(関数)"という構文で、変数を関数に投入
#「データセット名」[追加する列の名前] = 変数データ　という形で、新たな列を作ってそこにデータを投入
smalldf['noparking'] = smalldf.description.apply(noparking)

#疑問：.applyの前がここではsmalldf.descriptionだが、以前はsmalldf['link']だった。その違いは？
#基本的には同じ。smalldf.descriptionとsmalldf['description']は同じ。
#ドット記法とブラケット記法の違い。ドット記法は、Pythonの予約語（linkやclass）では使えない。ブラケット記法は何に対しても必ず使える。
#なので、ドット記法は書きやすさ重視、ブラケット記法は確実性重視、という違い。

#ここではdescriptionが、defの中とsmalldf.description.apply()の2箇所で出てくるが、2つは別物
#defの中のdescriptionは、引数。別にdescriptionでなくても何でもOK。関数の中だけで使われ、その定義はdefの外には残らない（forループでは最後のものが残る）
#後者はdata frameの列名。

In [3]:
# look at the output (just the noparking column)
smalldf.noparking

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
5      NaN
6      NaN
7      NaN
8      NaN
9    False
Name: noparking, dtype: object

This is a brute force method（総当り法） of parsing text. We are specifying all the combinations of text that might indicate that there are no parking spaces, and looking to see if any of them are contained in the string.

We'll see more sophisticated ways of analyzing text later in the course, but this type of approach is often the simplest and most robust.

<div class="alert alert-block alert-info">
<strong>Thought exercise:</strong> If you want to get the number of parking spaces for each project, what would be your next step? In principle, how might you do that?
</div>

<div class="alert alert-block alert-info">
<h3>Key Takeaways</h3>
<ul>
  <li>The simplest way to parse text is to look for a particular string within a longer string.</li>
  <li>Converting to lower case (or upper case) reduces the number of possibilities that you'll have to search for.
  <li>The <strong>in</strong> operator is most useful here.</li>
</ul>
</div>